## Great, now that we discussed a little let's continue

**Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.**

**Please explain, how we will achieve reproducibility for the course with this approach.**


We can achieve reproducibility by using standardised nf-core pipelines. As we saw the the first lecture of this course (in the day_01 folder), one of the main characteristics of the nf-core is that its workflows are designed to be reproducible. In addition, nf-core pipelines follow common standards and are regularly maintained, which makes them easier to run across different computing environments, including local machines, clusters and the cloud.

On the other hand, if we perform each step manually, we increase the probability of making mistakes and reduce the reproducibility of the analysis. Using nf-core helps us to follow the same workflow, tools and parameters, making it easier for other people to repeat the analysis, obtain comparable results and understand them.

**You have successfully downloaded 2 of the fastq files we will use in our study.**

**What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?**

The next step would be to process the FASTAQ files by using **nf-core/rnaseq**. This pipleine allows us to perform reads quality control, trimming and (pseudo-)alignment, and produces a gene count matrix and an extensive quality control report.

**Analyze the 2 files using an nf-core pipeline.**

**What does this pipeline do?**

**Which are the main tools that will be used in the pipeline?**

The pipeline processes the raw RNA-seq reads, checks the quality (FastQC), removes adapters (Trim Galore!), aligns the reads to the reference genome (STAR) and quantifies gene expression to produce a count matrix (Salmon) and quality control reports (MultiQC).

- FastQC: read quality control
- Trim Galore!: adapter and quality trimming
- STAR: alignment to the reference genome
- Salmon: quantifies gene/transcript expression
- SAMtools: process alignment files
- RSeQC/Qualimap: generate additional RNA-seq quality metrics
- MultiQC: combines QC results into a single report

**As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.**

**Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.**

**Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.**

In [13]:
# post here the command you used to run nf-core/rnaseq

import pandas as pd

samplesheet = pd.DataFrame({
    "sample": [
        "SNI-Sal-511",      #SRR23195511
        "Sham-Sal-516"      #SRR23195516
    ],
    "fastq_1": [
        "/home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195511_1.fastq.gz",
        "/home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195516_1.fastq.gz"
    ],
    "fastq_2": [
        "/home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195511_2.fastq.gz",
        "/home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195516_2.fastq.gz"
    ],
    "strandedness": [
        "auto",
        "auto"
    ]
})

samplesheet.to_csv("samplesheet.csv", index=False)

samplesheet


,sample,fastq_1,fastq_2,strandedness
0,SNI-Sal-511,/home/paula/.ssh/computational-workflows-2026/...,/home/paula/.ssh/computational-workflows-2026/...,auto
1,Sham-Sal-516,/home/paula/.ssh/computational-workflows-2026/...,/home/paula/.ssh/computational-workflows-2026/...,auto


In [14]:
import os

for f in samplesheet["fastq_1"].tolist() + samplesheet["fastq_2"].tolist():
    print(os.path.exists(f), f)

True /home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195511_1.fastq.gz
True /home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195516_1.fastq.gz
True /home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195511_2.fastq.gz
True /home/paula/.ssh/computational-workflows-2026/day_02/SRA_runs/fastq/SRR23195516_2.fastq.gz


In [ ]:
!nextflow run nf-core/rnaseq -r 3.27.0 -profile docker --input samplesheet.csv --outdir rnaseq_results4 --fasta references/chr19/Mus_musculus.GRCm38.dna.chromosome.19.fa.gz \
--gtf references/chr19/Mus_musculus.GRCm38.102.chr19.gtf -c /home/paula/.ssh/computational-workflows-2026/nextflow.config 


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/rnaseq` [grave_brattain] revision: a1fcdddd3b [3.27.0]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/rnaseq 3.27.0
------------------------------------------------------

Input/output options
  input              : samplesheet.csv
  outdir             : rnaseq_results4

Reference genome options
  genome             : GRCm38
  fasta              : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Sequence/WholeGenomeFasta/genome.fa
  gtf                : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Annotation/Genes/genes.gtf
  gene_bed           : s3://ngi-igenomes/igenomes//Mus_mu

**Explain all the parameters you set and why you set them in this way.**


- -r 3.27.0: selects this version of the nf-core/rnaseq pipeline so we can use the same version when repeating the analysis
- -profile docker: runs the pipeline inside Docker containers, which provide the required software and dependencies
- --input: specifies the samplesheet containing the sample names, FASTQ file paths and strandedness
- --outdir: specifies the output folder, where the results are going to be stored
- --fasta: specifies the reference DNA sequence used for alignment. Here we provide a file containing only the chromosome 19 of the mouse
- --gtf: specifies the matching annotation file, which describes the location of the genes, transcripts and exons of chromosome 19
- -c: specifies the configuration file, which sets resource limits such as memory and CPUs to keep the pipeline within the machine¡s available resources

## **Browsing the results**

**How did the pipeline perform?**

In my case, the pipeline did not finish because my computer ran out of storage while processing the data using the whole mouse genome reference. Some intermediate results were prodcued, but the final outputs were not complete. I later changed the command to use only the chromosome 19 to reduce resource requirements. However, I did not run the revised command or resume the pipeline because I was concerned about running out of storage again. Other students completed the pipeline sucessfully on their machines.

**Explain the quality control steps. Are you happy with the quality and why. If not, why not.**
**Please give additional information on:** 
- **ribosomal rRNA**
- **Duplication**
- **GC content**

**What are the possible steps that could lead to poorer results?**

In the RNAseq pipeline, quality controlis performed before and after processing the samples. In those step, FastQC are used to evaluate the quality of the sequencing reads. Before the processing, the samplesmay contain adapter contamination, low-quality bases, or other problems that can affect the analysis. After trimming with tools such as Trim Galore! or FastP, the quality of the reads can improve  because these tools remove adapter sequences and trim low-quality bases or reads. 

As mentioned before, I did not obtain the final results because my machine ran out of storage. Therefore, I cannot say whether I am happy with the quality of the samples. However, I would expect the quality after trimming to be better than before trimming if adapter contamination or low-quality regions were present.

**Ribosomal RNA (rRNA)** is a type of non-coding RNA. To create the structure of the ribosomes, rRNAs should combine with proteins. At the same time that ribosomes read the instructions that are encoded in the mRNAs, the rRNA are the responsibles of the chemical reactions that assemble the new proteins piece-by-piece. Moreover, it is worth of mention, rRNAs are about 80-90% af all the RNA in a human cell. (https://www.genome.gov/about-genomics/educational-resources/fact-sheets/ribonucleic-acid-fact-sheet). Therefore, it is important to remove or reduce rRNA in many RNA-seq experiments, since otherwise a large portion of the sequencing reads may come from rRNA instead of the transcripts we are interested in analysing.

If too many reads correspond to rRNA, fewer reads are available for low-abundance mRNAs and other trasncripts. In nf-core/rnaseq, rRNA removal can be anabled with the `--remove_ribo_rna` parameter using tools such as SortMeRNA. 

**Duplication** refers to the presence ofmany identical or very similar reads in the sequencing data. During library preparation, the RNA is fragmented and the resulting fragments are amplified by PCR. If the same fragment is amplified too many times, we can end up sequencing several copies of the same original fragment. This can be a problem for quality control because it may give the impression that we have more independent information than we actually do. In reality, many of those reads may come from the same original molecule. High duplication can therefore reduce the complexity of the library and may also affect the estimation of how abundant some genes are.

However, some duplication is normal in RNA-seq, especially for genes that are highly expressed. For this reason, a high duplication level does not automatically mean that the sample has poor quality, but it can be a warning sign that should be interpreted together with the other quality control results. It is not always possible to know whether duplicated reads come from PCR amplification or from genuinely highly expressed genes. For this reason, duplication levels should be interpreted together with other quality control metrics.

**GC content** refers to the proportion of G and C bases present in the sequencing reads. In quality control, we look at if the overall content of GC content of the reads follows the pattern that we would expect for that type of sample. Iff the GC content is very different from what is expected, or if the distribution shows unusual peaks, this can be a warining sign that something may be affecting the sample. For example, it could be related to contamination, biases introduced during the library preparation, or the presence of sequences that are not representative of the sample we wanted to analyse.

For this reason, GC content is useful as a general quality control check. A normal distribution does not guarantee that the sample is perfect, but an unusual distribution can help identify possible problems that should be investigated further.

Since I could not obtain the results, I cannot evaluate whether the G content of my samples followed the expected distribution. 

**Would you exclude any samples? If yes, which and why?**

Since I did not obtain the quality control results, I cannot determine whether any samples should be excluded from the analysis. To make that decision, I would need to look at the QC results and check whether any samples shows clear problems, such as very low read quality, high adapter contamination, unusual GC content, excessive duplication, or another abnormal patterns compared with the rest of the samples

If one sample showed consistently poor quality across several QC metrics, I would consider excluding it from the downstream analysis. However, without the results, I cannot identify any specific sample that should be removed.

**What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?**

To continue with the experiment, I would first process the reads further by aligning or quantifying them and then generate a gene count matrix. After that, I would compare the different experimental conditions to identify genes that are expressed differently between the groups.

The main goal of the experiemtn is to understand how the gene expression changes depending on the experimental condition (Sham vs. SNI, and saline vs OXY). This could help identify genes and biological pathways that are affected by the injury or by the treatment.

For downstream analysis, I would mainly use R because it is widely used for statistical analysis and data visualisation. This is specially useful in RNA-seq, where we need to compare gene expression between different experimental groups and determine whether the observed differences are statistically significant.

In R, I would use different packages depending on the analysis. `DESeq2` can be used for differential expression analysis, `ggplot2` for data visualisation, `pheatmap` or `ComplexHeatmap` for creating heatmaps, and `clusterProfiler` for investigating whether the differentially expressed genes are associated with specific biological pathways or functions.  